# RSO-901 Glycol Set Points — Command Response

How quickly, and how reliably, do the two EAS-controlled glycol chillers (Chiller 01 and Chiller 02)
follow the set points that EAS commands? This compares every `HVAC.configChiller` command with the
chiller's own reported set point over an arbitrary `day_obs` range (set `start_day_obs` and
`end_day_obs` below, both integers `YYYYMMDD`; the window runs from 12:00 UTC on the first `day_obs`
through 11:59:59 UTC the day after the last one).

It answers three questions:

1. **Latency** — when EAS commands a *new* value, how long until the chiller reports it, and how often
   does it never do so?
2. **Mismatch** — when EAS *repeats* a value (it re-sends every 60 s), how often is the chiller's
   reported set point not that value, and for how long?
3. **Recovery** — when a mismatch ends, was it fixed by re-sending the same value, or only by sending a
   different one or by the chiller restarting?
4. **Size** — how far off is the chiller during a mismatch? Most mismatches are a fraction of a degree
   and do not matter, so the size section picks out the *large* ones (`LARGE_MISMATCH_C` and above) and
   shows how big they are and how long they last.

**Why these topics.** `command_configChiller` is what EAS asked for. `logevent_chillerConfiguration`
looks like a record of the commanded value, but the HVAC CSC builds it from the chiller's *telemetry*
(`_send_config_event` in `ts_hvac`), and only while the chiller is switched on. It is therefore the
chiller's **readback**, written only when the readback changes, which makes it cheap to query over many
months. `logevent_deviceEnabled` says when each chiller was switched off, so readbacks are not judged
while the chiller is off.

Method notes: the chiller truncates set points *downward* to 0.1 °C (commanded 6.18 reads back 6.1),
so a readback "matches" a command when it is within 0.11 °C and not above it. Commands are only judged
once `RESPONSE_WINDOW_S` seconds have passed since the commanded value last changed.

See `ai_notes.md` in this folder for the topic survey and field reference. This is the command-side
companion to the *Date Range* notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# The getDayObs* helpers live in dateTime; importing them from efdUtils is
# deprecated and those shims are removed after w_2026_01.
from lsst.summit.utils.dateTime import getDayObsStartTime, getDayObsEndTime
from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.ts.xml.enums.HVAC import DeviceId

## Parameters

In [ ]:
# Inclusive day_obs range, as integers YYYYMMDD. The observatory day rolls over at
# 12:00 UTC (UTC-12 convention), so the window is 12:00 UTC on start_day_obs through
# 11:59:59 UTC the day after end_day_obs.
start_day_obs = 20260101
end_day_obs = 20260720  # storm night

# device_id (DeviceId_chiller enum) -> name. Only the two chillers EAS commands.
CHILLERS = {
    101: "chiller01",
    102: "chiller02",
}

COMMAND_TOPIC = "lsst.sal.HVAC.command_configChiller"  # what EAS asked for
READBACK_TOPIC = "lsst.sal.HVAC.logevent_chillerConfiguration"  # what the chiller reports
ENABLED_TOPIC = "lsst.sal.HVAC.logevent_deviceEnabled"  # bitmask of switched-on devices

# A changed command counts as answered if the readback matches within this many seconds, and a
# command is only judged for mismatch once its value is at least this old.
RESPONSE_WINDOW_S = 60.0

# The chiller truncates downward to 0.1 degC, so a readback matches if it is within this of the command
# and not above it.
READBACK_TOLERANCE = 0.11

# A mismatch counts as "large" when the readback is at least this far from the command (degC, either
# direction). Smaller mismatches still appear in the all-mismatch numbers but are not of interest in the
# size section.
LARGE_MISMATCH_C = 0.5

# Query the EFD this many day_obs at a time, so a multi-month range does not time out.
CHUNK_DAYS = 10

# Curated CSV of catastrophic glycol / camera-cooling events, overlaid on the mismatch timeline.
EVENTS_CSV = "glycol_catastrophic_faults.csv"

# Position of each chiller in the deviceEnabled bitmask: the HVAC CSC numbers devices by their
# order in the DeviceId enum.
ENABLED_BIT = {dev: list(DeviceId).index(DeviceId(dev)) for dev in CHILLERS}

In [ ]:
if end_day_obs < start_day_obs:
    raise ValueError(f"end_day_obs {end_day_obs} < start_day_obs {start_day_obs}")

efd_client = makeEfdClient()

begin = getDayObsStartTime(start_day_obs)  # 12:00 UTC on start_day_obs
end = getDayObsEndTime(end_day_obs)  # 11:59:59 UTC the day after end_day_obs

print(f"day_obs {start_day_obs}\u2013{end_day_obs}: {begin.isot} -> {end.isot} (UTC)")

## Query the commands, readbacks and enabled mask

Every query is split into `CHUNK_DAYS`-day pieces. The commands are the dense part: EAS sends one per
chiller per minute, about 580 000 rows over six months. The readback and enabled events are sparse, so
they are cheap.

In [ ]:
days = pd.date_range(str(start_day_obs), str(end_day_obs), freq="D")
chunks = [days[i : i + CHUNK_DAYS] for i in range(0, len(days), CHUNK_DAYS)]


def day_obs_of(timestamp):
    return int(timestamp.strftime("%Y%m%d"))


def query_chunked(topic, **kwargs):
    frames = []
    for chunk in chunks:
        part = getEfdData(
            efd_client,
            topic,
            begin=getDayObsStartTime(day_obs_of(chunk[0])),
            end=getDayObsEndTime(day_obs_of(chunk[-1])),
            **kwargs,
        )
        if len(part):
            frames.append(part)
    return pd.concat(frames).sort_index() if frames else pd.DataFrame()


def to_naive_utc(frame):
    # Drop tz (values are UTC) so everything compares against the naive window and events below.
    if len(frame) and frame.index.tz is not None:
        frame = frame.copy()
        frame.index = frame.index.tz_convert("UTC").tz_localize(None)
    return frame


raw_commands = to_naive_utc(
    query_chunked(COMMAND_TOPIC, columns=["activeSetpoint", "device_id", "private_identity"])
)
raw_readbacks = to_naive_utc(query_chunked(READBACK_TOPIC, columns=["activeSetpoint", "device_id"]))
raw_enabled = to_naive_utc(query_chunked(ENABLED_TOPIC, columns=["device_ids"]))

print(f"{len(raw_commands)} command(s), {len(raw_readbacks)} readback event(s), "
      f"{len(raw_enabled)} deviceEnabled event(s)")
raw_commands.head() if len(raw_commands) else raw_commands

## Line up commands with readbacks

For each chiller this attaches to every command the readback in force when it arrived, whether the
chiller was switched on, and how long ago the commanded value last changed. A command that merely
repeats the previous value is `changed = False`; the first command in the range is not a change either,
because there is nothing to compare with.

`deviceEnabled` is an event, so the state at the start of the range is unknown and the chiller is
assumed to be on until the first event says otherwise.

In [ ]:
def matches(readback, command):
    """True where the readback is the command as the chiller reports it (truncated down to 0.1)."""
    return (np.abs(readback - command) <= READBACK_TOLERANCE) & (readback <= command + 1e-6)


def asof(index, series):
    """Value of a step series in force at each timestamp of ``index`` (NaN before its first row)."""
    series = series.sort_index()
    if not len(series):
        return np.full(len(index), np.nan)
    pos = series.index.searchsorted(index, side="right") - 1
    values = series.to_numpy(dtype=float)[np.clip(pos, 0, None)]
    return np.where(pos >= 0, values, np.nan)


commands, readbacks, enabled_state = {}, {}, {}
for dev in CHILLERS:
    readbacks[dev] = raw_readbacks.loc[raw_readbacks["device_id"] == dev, "activeSetpoint"].sort_index()

    if len(raw_enabled):
        bits = raw_enabled["device_ids"].astype("int64").to_numpy() >> ENABLED_BIT[dev] & 1
        enabled_state[dev] = pd.Series(bits, index=raw_enabled.index).sort_index()
    else:
        enabled_state[dev] = pd.Series(dtype=float)

    c = raw_commands.loc[raw_commands["device_id"] == dev].sort_index().copy()
    c["changed"] = c["activeSetpoint"].diff().abs().gt(1e-6)  # first row: NaN diff -> False
    c["readback"] = asof(c.index, readbacks[dev])
    c["enabled"] = np.nan_to_num(asof(c.index, enabled_state[dev]), nan=1.0).astype(bool)
    c["matched"] = matches(c["readback"], c["activeSetpoint"])
    c["error"] = c["readback"] - c["activeSetpoint"]  # negative: chiller colder than commanded

    # Age of the commanded value: seconds since it last changed (or since the first command).
    stamp = c.index.to_series()
    last_change = stamp.where(c["changed"] | (stamp == stamp.iloc[0])).ffill() if len(c) else stamp
    c["age_s"] = (stamp - last_change).dt.total_seconds()
    commands[dev] = c

for dev, name in CHILLERS.items():
    c = commands[dev]
    print(f"{name}: {len(c)} commands, {int(c['changed'].sum())} changed, "
          f"{len(readbacks[dev])} readback events, "
          f"{int((~c['enabled']).sum())} commands sent while the chiller was off")

if len(raw_commands) and "private_identity" in raw_commands:
    print("\nCommand senders:")
    print(raw_commands["private_identity"].value_counts())

## 1. Response to a new command

For each command whose value differs from the previous one, find the first readback event within
`RESPONSE_WINDOW_S` that matches it. Each command gets one status:

| Status | Meaning |
|---|---|
| `responded` | The readback matched within the window; `latency_s` is the delay. |
| `no_response` | The window passed without a matching readback. |
| `already_matched` | The readback already matched when the command arrived (for example a 0.05 °C change that truncates to the same value), so no event is expected. |
| `chiller_off` | The chiller was switched off, so no readback is expected. |

In [ ]:
window = pd.Timedelta(seconds=RESPONSE_WINDOW_S)


def first_match_latency(series, t, target):
    lo = series.index.searchsorted(t, side="left")
    hi = series.index.searchsorted(t + window, side="right")
    for stamp, value in zip(series.index[lo:hi], series.iloc[lo:hi]):
        if matches(value, target):
            return (stamp - t).total_seconds()
    return np.nan


rows = []
for dev, name in CHILLERS.items():
    c = commands[dev]
    previous = c["activeSetpoint"].shift()
    for t, row in c[c["changed"]].iterrows():
        if not row["enabled"]:
            status, latency = "chiller_off", np.nan
        elif row["matched"]:
            status, latency = "already_matched", np.nan
        else:
            latency = first_match_latency(readbacks[dev], t, row["activeSetpoint"])
            status = "responded" if not np.isnan(latency) else "no_response"
        rows.append(
            dict(
                time=t,
                chiller=name,
                old_command=previous[t],
                command=row["activeSetpoint"],
                readback_before=row["readback"],
                status=status,
                latency_s=latency,
            )
        )

latency = pd.DataFrame(rows)
if len(latency):
    print(latency.groupby(["chiller", "status"]).size().unstack(fill_value=0))
    print()
    print(latency.groupby("chiller")["latency_s"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))
    display(latency[latency["status"] == "no_response"])
else:
    print("No changed commands in this range.")

## Figure — response latency

Left: distribution of the delay between a new command and the matching readback. Right: the same
delays through time, with unanswered commands drawn as red crosses at the top of the window.

In [ ]:
stem = f"chiller_command_response_{start_day_obs}_{end_day_obs}"
os.makedirs("plots", exist_ok=True)

if len(latency):
    fig, (ax_hist, ax_time) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw=dict(width_ratios=[1, 2]))
    bins = np.linspace(0, RESPONSE_WINDOW_S, int(RESPONSE_WINDOW_S) + 1)

    for name in CHILLERS.values():
        g = latency[latency["chiller"] == name]
        ok = g[g["status"] == "responded"]
        ax_hist.hist(ok["latency_s"], bins=bins, alpha=0.6, label=f"{name} (n={len(ok)})")
        ax_time.plot(ok["time"], ok["latency_s"], "o", ms=3, alpha=0.7, label=name)
        lost = g[g["status"] == "no_response"]
        ax_time.plot(lost["time"], np.full(len(lost), RESPONSE_WINDOW_S), "x", color="#E5484D", ms=8)

    ax_hist.set_xlabel("Latency to matching readback [s]")
    ax_hist.set_ylabel("Commands")
    ax_hist.legend()
    ax_time.set_ylabel("Latency [s]")
    ax_time.set_xlabel("Time [UTC]")
    ax_time.set_ylim(0, RESPONSE_WINDOW_S * 1.05)
    ax_time.legend()
    for ax in (ax_hist, ax_time):
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"Chiller response to a new set point — day_obs {start_day_obs}\u2013{end_day_obs}")
    fig.tight_layout()
    fig.savefig(f"plots/{stem}_latency.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Nothing to plot.")

## 2. Mismatch between command and readback

Every command (new or repeated) is judged against the readback in force when it arrived, but only when

* the chiller was switched on,
* a readback exists, and
* the commanded value is older than `RESPONSE_WINDOW_S`, so a chiller still on its way to a new value
  is not counted.

A **mismatch episode** is a run of consecutive judged commands that all fail to match. It ends at the
first judged command that matches again, or stays *unresolved* if the range ends first. Commands sent
while the chiller was off are skipped, so an episode can span a power cycle, and `off_during` flags
those.

In [ ]:
def find_episodes(judged, flag, name, all_commands):
    """Runs of consecutive judged commands where ``flag`` is True, one row per run."""
    rows = []
    run = (judged[flag] != judged[flag].shift()).cumsum()
    for _, g in judged[judged[flag]].groupby(run[judged[flag]]):
        start = g.index[0]
        after = judged.loc[judged.index > g.index[-1]]
        resolved = after[~after[flag]]
        stop = resolved.index[0] if len(resolved) else g.index[-1]
        first_cmd, last_cmd = g["activeSetpoint"].iloc[0], g["activeSetpoint"].iloc[-1]
        if not len(resolved):
            resolved_by = "unresolved"
        elif g["activeSetpoint"].nunique() == 1 and np.isclose(resolved["activeSetpoint"].iloc[0], first_cmd):
            resolved_by = "same value re-sent"
        else:
            resolved_by = "new command value"
        off_during = bool((~all_commands.loc[start:stop, "enabled"]).any())
        if off_during and resolved_by == "same value re-sent":
            resolved_by = "chiller restarted, same value"  # the restart, not the repeat, may have fixed it
        worst = g["error"].abs().idxmax()
        rows.append(
            dict(
                chiller=name,
                start=start,
                end=stop,
                duration_min=(stop - start).total_seconds() / 60,
                n_commands=len(g),
                command_first=first_cmd,
                command_last=last_cmd,
                readback_first=g["readback"].iloc[0],
                readback_last=g["readback"].iloc[-1],
                max_abs_error_c=g["error"].abs().max(),
                worst_error_c=g.loc[worst, "error"],
                off_during=off_during,
                resolved_by=resolved_by,
            )
        )
    return rows


episode_rows, large_rows, summary_rows, judged_parts = [], [], [], []
for dev, name in CHILLERS.items():
    c = commands[dev]
    judged = c[c["enabled"] & c["readback"].notna() & (c["age_s"] > RESPONSE_WINDOW_S)].copy()
    judged["mismatch"] = ~judged["matched"]
    judged["large"] = judged["mismatch"] & (judged["error"].abs() >= LARGE_MISMATCH_C)
    judged_parts.append(judged.assign(chiller=name))

    summary_rows.append(
        dict(
            chiller=name,
            judged=len(judged),
            mismatched=int(judged["mismatch"].sum()),
            mismatch_pct=100 * judged["mismatch"].mean() if len(judged) else np.nan,
            large=int(judged["large"].sum()),
            large_pct=100 * judged["large"].mean() if len(judged) else np.nan,
        )
    )
    episode_rows += find_episodes(judged, "mismatch", name, c)
    large_rows += find_episodes(judged, "large", name, c)

judged_all = pd.concat(judged_parts)
episodes = pd.DataFrame(episode_rows)
large_episodes = pd.DataFrame(large_rows)

summary = pd.DataFrame(summary_rows).set_index("chiller")
for label, eps in (("episodes", episodes), ("large_episodes", large_episodes)):
    if len(eps):
        summary[label] = eps.groupby("chiller").size()
        summary[label.replace("episodes", "hours")] = eps.groupby("chiller")["duration_min"].sum() / 60
summary = summary.fillna(0)
print(f"All mismatches (readback off by more than {READBACK_TOLERANCE} degC) and large ones "
      f"(off by {LARGE_MISMATCH_C} degC or more):")
print(summary.round(3))
print()
if len(episodes):
    print(episodes.groupby(["chiller", "resolved_by", "off_during"]).size().rename("episodes"))
else:
    print("No mismatch episodes in this range.")

## Figure — mismatch timeline

One row per chiller, with a bar for each mismatch episode: grey for episodes that never reached
`LARGE_MISMATCH_C`, coloured for those that did. Dashed vertical lines are the catastrophic cooling
events from `glycol_catastrophic_faults.csv` (red for realised faults, amber for near-misses), tagged
`#N` as in the *Date Range* notebook. The right panel is the distribution of episode lengths.

In [ ]:
EVENT_COLORS = {"actual": "#E5484D", "near_miss": "#F5A623"}
CHILLER_COLORS = {"chiller01": "#00C2C2", "chiller02": "#FF7A00"}

events = pd.read_csv(EVENTS_CSV)
events["t"] = pd.to_datetime(events["fault_start_timestamp_utc"], utc=True).dt.tz_localize(None)
win_lo, win_hi = begin.utc.datetime, end.utc.datetime
events_win = events[(events["t"] >= win_lo) & (events["t"] <= win_hi)].sort_values("t")
print(f"{len(events_win)} catastrophic event(s) within the window")


def draw_episodes(ax, eps, y, **kwargs):
    # Bars are placed in matplotlib date numbers, with a little minimum width so one-command episodes
    # stay visible on a months-long axis.
    if not len(eps):
        return
    widths = (eps["end"] - eps["start"]).clip(lower=pd.Timedelta(hours=2))
    start_num = mdates.date2num(eps["start"])
    ax.barh(y, mdates.date2num(eps["start"] + widths) - start_num, left=start_num, height=0.6, **kwargs)


fig, (ax, ax_dur) = plt.subplots(1, 2, figsize=(15, 3.5), gridspec_kw=dict(width_ratios=[3, 1]))
for i, name in enumerate(CHILLERS.values()):
    is_large = episodes["max_abs_error_c"] >= LARGE_MISMATCH_C if len(episodes) else None
    mine = episodes["chiller"] == name if len(episodes) else None
    if len(episodes):
        draw_episodes(ax, episodes[mine & ~is_large], i, color="#b8b7b1")
        draw_episodes(ax, episodes[mine & is_large], i, color=CHILLER_COLORS[name])
ax.xaxis_date()
ax.set_xlim(win_lo, win_hi)
ax.set_ylim(-0.6, len(CHILLERS) - 0.4)  # fixed, so a chiller with no episodes keeps its row
ax.set_yticks(range(len(CHILLERS)))
ax.set_yticklabels(list(CHILLERS.values()))
ax.set_xlabel("Time [UTC]")
ax.set_title(f"Mismatch episodes (grey: never reached {LARGE_MISMATCH_C} \u00b0C; colour: reached it)",
             loc="left", fontsize=10)

for _, ev in events_win.iterrows():
    color = EVENT_COLORS.get(ev["outcome"], "#888888")
    ax.axvline(ev["t"], color=color, ls="--", lw=1.2)
    ax.annotate(f"#{int(ev['event_id'])}", (ev["t"], len(CHILLERS) - 0.45), color=color, fontsize=8,
                ha="center", va="bottom", annotation_clip=False)

if len(episodes):
    top = max(episodes["duration_min"].max(), 1)
    ax_dur.hist(episodes["duration_min"], bins=np.logspace(-1, np.log10(top * 1.2), 25))
    ax_dur.set_xscale("log")
ax_dur.set_xlabel("Episode length [min]")
ax_dur.set_ylabel("Episodes")
for a in (ax, ax_dur):
    a.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"plots/{stem}_mismatch.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. How large are the mismatches?

Most mismatches are small, so this section looks at how far off the chiller is, using the error
`readback − command` (negative means the chiller is colder than commanded). The table counts judged
commands by the size of that error; since EAS sends one command per chiller per minute, a count is about
the same number of minutes. Then the **large** mismatches (`|error| ≥ LARGE_MISMATCH_C`) are listed as
episodes: the run of consecutive judged commands that stay large. A large episode ends as soon as the
error falls below the threshold, even if a small mismatch remains.

In [ ]:
size_bins = [READBACK_TOLERANCE, 0.25, 0.5, 1, 2, 5, np.inf]
size_labels = ["0.11\u20130.25", "0.25\u20130.5", "0.5\u20131", "1\u20132", "2\u20135", ">5"]

mismatched = judged_all[judged_all["mismatch"]].copy()
mismatched["size_c"] = pd.cut(mismatched["error"].abs(), size_bins, labels=size_labels)
print("Mismatched commands by |readback - command| [degC] (about the same number of minutes):")
print(mismatched.groupby(["chiller", "size_c"], observed=False).size().unstack(fill_value=0))
print()

for name in CHILLERS.values():
    m = mismatched[mismatched["chiller"] == name]
    if not len(m):
        print(f"{name}: no mismatches")
        continue
    big = m[m["large"]]
    print(f"{name}: {len(m)} mismatched commands, {len(big)} large ({100 * len(big) / len(m):.0f}% of the "
          f"mismatches); median |error| {m['error'].abs().median():.2f} degC, "
          f"largest {m['error'].abs().max():.2f} degC")

print()
if len(large_episodes):
    print(f"{len(large_episodes)} large episode(s), {large_episodes['duration_min'].sum() / 60:.1f} h in total:")
    print(large_episodes.groupby(["chiller", "resolved_by", "off_during"]).size().rename("episodes"))
    print()
    display(large_episodes.sort_values("max_abs_error_c", ascending=False).reset_index(drop=True))
else:
    print(f"No mismatch reached {LARGE_MISMATCH_C} degC in this range.")

## Figure — size of the mismatches

Four views of the same mismatching commands. **Top left:** the readback against the command, with the
`±LARGE_MISMATCH_C` band shaded; points on the diagonal agree. **Top right:** the error through time.
**Bottom left:** how many mismatching commands fall at each error size. **Bottom right:** one dot per
episode, its longest duration against its worst error, so short and small episodes can be told apart
from long and large ones.

In [ ]:
if len(mismatched):
    fig, ((ax_xy, ax_t), (ax_h, ax_ep)) = plt.subplots(2, 2, figsize=(14, 9))

    lims = [mismatched[["activeSetpoint", "readback"]].min().min() - 1,
            mismatched[["activeSetpoint", "readback"]].max().max() + 1]
    xs = np.array(lims)
    ax_xy.fill_between(xs, xs - LARGE_MISMATCH_C, xs + LARGE_MISMATCH_C, color="0.9", label=f"\u00b1{LARGE_MISMATCH_C} \u00b0C")
    ax_xy.plot(xs, xs, color="0.5", lw=1)
    for name, color in CHILLER_COLORS.items():
        m = mismatched[mismatched["chiller"] == name]
        if not len(m):
            continue
        ax_xy.plot(m["activeSetpoint"], m["readback"], ".", ms=4, alpha=0.4, color=color, label=name)
        ax_t.plot(m.index, m["error"], ".", ms=4, alpha=0.4, color=color, label=name)
        ax_h.hist(m["error"].abs(), bins=np.logspace(np.log10(READBACK_TOLERANCE), np.log10(max(m["error"].abs().max(), 1) * 1.2), 30),
                  alpha=0.6, color=color, label=name)
        e = large_episodes[large_episodes["chiller"] == name] if len(large_episodes) else large_episodes
        small = episodes[(episodes["chiller"] == name) & (episodes["max_abs_error_c"] < LARGE_MISMATCH_C)]
        ax_ep.plot(small["duration_min"].clip(lower=0.5), small["max_abs_error_c"], "o", ms=5, alpha=0.5,
                   color="#b8b7b1", mfc="none")
        if len(e):
            ax_ep.plot(e["duration_min"].clip(lower=0.5), e["max_abs_error_c"], "o", ms=7, alpha=0.8, color=color,
                       label=name)

    ax_xy.set_xlim(lims)
    ax_xy.set_ylim(lims)
    ax_xy.set_xlabel("Commanded set point [\u00b0C]")
    ax_xy.set_ylabel("Chiller readback [\u00b0C]")
    ax_xy.set_title("Mismatching commands: readback vs command", loc="left", fontsize=10)
    ax_xy.legend()

    for sign in (-1, 1):
        ax_t.axhline(sign * LARGE_MISMATCH_C, color="0.5", ls="--", lw=1)
    ax_t.axhline(0, color="0.7", lw=1)
    ax_t.set_xlabel("Time [UTC]")
    ax_t.set_ylabel("Readback \u2212 command [\u00b0C]")
    ax_t.set_title("Error through time (dashed: large-mismatch threshold)", loc="left", fontsize=10)
    ax_t.legend()

    ax_h.axvline(LARGE_MISMATCH_C, color="0.5", ls="--", lw=1)
    ax_h.set_xscale("log")
    ax_h.set_yscale("log")
    ax_h.set_xlabel("|readback \u2212 command| [\u00b0C]")
    ax_h.set_ylabel("Mismatching commands")
    ax_h.set_title("Size of the mismatches", loc="left", fontsize=10)
    ax_h.legend()

    ax_ep.axhline(LARGE_MISMATCH_C, color="0.5", ls="--", lw=1)
    ax_ep.set_xscale("log")
    ax_ep.set_xlabel("Episode length [min]")
    ax_ep.set_ylabel("Worst |error| in episode [\u00b0C]")
    ax_ep.set_title("Episodes (hollow grey: never large)", loc="left", fontsize=10)
    if ax_ep.get_legend_handles_labels()[0]:
        ax_ep.legend()

    for a in (ax_xy, ax_t, ax_h, ax_ep):
        a.grid(True, alpha=0.3)
    fig.suptitle(f"Chiller mismatch size \u2014 day_obs {start_day_obs}\u2013{end_day_obs}")
    fig.tight_layout()
    fig.savefig(f"plots/{stem}_mismatch_size.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No mismatching commands to plot.")

## Summary by month

Per chiller and calendar month: how many new commands there were and how many were answered, the median
latency, the share of judged commands that did not match the readback (all mismatches and large ones),
and the hours spent in large-mismatch episodes. A month where `answered` drops or `large_pct` jumps is
worth checking against the events table.

In [ ]:
parts = []
if len(latency):
    lat = latency.assign(month=latency["time"].dt.to_period("M"))
    lat_ok = lat[~lat["status"].isin(["already_matched", "chiller_off"])]
    parts.append(lat_ok.groupby(["chiller", "month"]).agg(
        new_commands=("status", "size"),
        answered_pct=("status", lambda s: 100 * (s == "responded").mean()),
        median_latency_s=("latency_s", "median"),
    ))

by_month = judged_all.assign(month=judged_all.index.to_period("M"))
parts.append(by_month.groupby(["chiller", "month"]).agg(
    mismatch_pct=("mismatch", lambda s: 100 * s.mean()),
    large_pct=("large", lambda s: 100 * s.mean()),
))

if len(large_episodes):
    ep = large_episodes.assign(month=large_episodes["start"].dt.to_period("M"))
    parts.append(ep.groupby(["chiller", "month"]).agg(
        large_episodes=("duration_min", "size"),
        large_hours=("duration_min", lambda s: s.sum() / 60),
    ))

monthly = pd.concat(parts, axis=1)
display(monthly.round(2))

## Mismatch episodes

All episodes (small ones included), longest first. The large ones are already listed above. The columns
to read are `resolved_by` and `off_during`:

* `same value re-sent` means the chiller caught up while EAS kept sending the *same* value and the
  chiller was never switched off. That is the normal recovery path if the chiller simply applies what
  it is sent.
* `chiller restarted, same value` means the mismatch ended while EAS was still sending the same value,
  but the chiller had been switched off during the episode, so the restart may be what fixed it.
* `new command value`, or `unresolved`, with the same value repeated for most of the episode means the
  repeats did nothing and only a different value (or a restart) cleared the mismatch.

In [ ]:
if len(episodes):
    display(episodes.sort_values("duration_min", ascending=False).reset_index(drop=True))
else:
    print("No mismatch episodes in this range.")